# StaleBench demo

Measure how *stale* a RAG system's answers get after a fact changes. This runs against any OpenAI-compatible model endpoint (OpenAI, OpenRouter, vLLM, Ollama, LM Studio, ...).

Fill in your endpoint and key below, then run all cells.

In [ ]:
!pip install -q git+https://github.com/KaranSinghDev/StaleBench.git sentence-transformers

In [ ]:
from stalebench import make_scenario, benchmark
from stalebench.reference import ReferenceRAG
from stalebench.backends import OpenAICompatLLM

# Point at any OpenAI-compatible endpoint and fill in your details:
BASE_URL = "https://openrouter.ai/api/v1"            # your endpoint
API_KEY  = "YOUR_API_KEY"                            # your key
MODEL    = "meta-llama/llama-3.2-3b-instruct"        # any chat model on that endpoint

llm = OpenAICompatLLM(model=MODEL, base_url=BASE_URL, api_key=API_KEY, temperature=0)
rag = ReferenceRAG(llm=llm, retriever="tfidf", top_k=3)
scenario = make_scenario(n_facts=12, seed=0)

aggregate, _ = benchmark(rag, scenario, trials=2)
for policy, s in aggregate.items():
    print(f"{policy:10s} recovery={s['rate']:.2f} CI{s['rate_ci']}")

**Reading the output.** `never` should be about 0 (sanity check). Under `immediate` refresh, the share of answers that still stay stale is the freshness gap.

**Try the fix.** Re-create the system with `ReferenceRAG(llm=llm, retriever="tfidf", top_k=3, recency_order=True)` to place the newest document last, and see whether recovery goes up (it helps some models and backfires on others).

To benchmark **your own** RAG, subclass `stalebench.RAGSystem` with `index(documents)` and `answer(query)` methods. See the project README.